# 01 — Data Extraction & Profiling

**Sector:** E-commerce  |  **Source:** [Kaggle — Amazon Products Sales Dataset (42K, 2025)](https://www.kaggle.com/datasets/ikramshah512/amazon-products-sales-dataset-42k-items-2025)
**Raw file:** `data/raw/amazon_products_sales_data_uncleaned.csv`

Goals:
1. Load the raw CSV; confirm shape & column list.
2. Profile dtypes, missingness, and obvious quality issues.
3. Document the parsing rules to apply in `02_cleaning.ipynb`.

> **Capstone rule:** never modify the raw file.

In [1]:
from pathlib import Path
import pandas as pd

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW = ROOT / "data" / "raw" / "amazon_products_sales_data_uncleaned.csv"
print("Project root:", ROOT)
print("Raw file:", RAW, "  exists:", RAW.exists())

Project root: /Users/aashishjha/Documents/Projects/DVA-capstone-2
Raw file: /Users/aashishjha/Documents/Projects/DVA-capstone-2/data/raw/amazon_products_sales_data_uncleaned.csv   exists: True


## 1. Load raw

In [2]:
df = pd.read_csv(RAW, low_memory=False)
print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} cols")
df.head(3)

Shape: 42,675 rows × 16 cols


,title,rating,number_of_reviews,bought_in_last_month,current/discounted_price,price_on_variant,listed_price,is_best_seller,is_sponsored,is_couponed,buy_box_availability,delivery_details,sustainability_badges,image_url,product_url,collected_at
0,BOYA BOYALINK 2 Wireless Lavalier Microphone f...,4.6 out of 5 stars,375,300+ bought in past month,89.68,basic variant price: 2.4GHz,$159.00,No Badge,Sponsored,Save 15% with coupon,Add to cart,"Delivery Mon, Sep 1",Carbon impact,https://m.media-amazon.com/images/I/71pAqiVEs3...,/sspa/click?ie=UTF8&spc=MTo4NzEzNDY2NTQ5NDYxND...,2025-08-21 11:14:29
1,"LISEN USB C to Lightning Cable, 240W 4 in 1 Ch...",4.3 out of 5 stars,"2,457",6K+ bought in past month,9.99,basic variant price: nan,$15.99,No Badge,Sponsored,No Coupon,Add to cart,"Delivery Fri, Aug 29",NaN,https://m.media-amazon.com/images/I/61nbF6aVIP...,/sspa/click?ie=UTF8&spc=MTo4NzEzNDY2NTQ5NDYxND...,2025-08-21 11:14:29
2,"DJI Mic 2 (2 TX + 1 RX + Charging Case), Wirel...",4.6 out of 5 stars,"3,044",2K+ bought in past month,314.00,basic variant price: nan,$349.00,No Badge,Sponsored,No Coupon,Add to cart,"Delivery Mon, Sep 1",NaN,https://m.media-amazon.com/images/I/61h78MEXoj...,/sspa/click?ie=UTF8&spc=MTo4NzEzNDY2NTQ5NDYxND...,2025-08-21 11:14:29


## 2. Schema & dtypes

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 42675 entries, 0 to 42674
Data columns (total 16 columns):
 #   Column                    Non-Null Count  Dtype
---  ------                    --------------  -----
 0   title                     42675 non-null  str  
 1   rating                    41651 non-null  str  
 2   number_of_reviews         41651 non-null  str  
 3   bought_in_last_month      39458 non-null  str  
 4   current/discounted_price  30926 non-null  str  
 5   price_on_variant          42675 non-null  str  
 6   listed_price              42675 non-null  str  
 7   is_best_seller            42675 non-null  str  
 8   is_sponsored              42675 non-null  str  
 9   is_couponed               42675 non-null  str  
 10  buy_box_availability      28022 non-null  str  
 11  delivery_details          30955 non-null  str  
 12  sustainability_badges     3408 non-null   str  
 13  image_url                 42675 non-null  str  
 14  product_url               40606 non-null  str  
 

## 3. Missingness

In [4]:
missing = (
    df.isna().sum().to_frame("missing")
    .assign(pct=lambda x: (x["missing"] / len(df) * 100).round(2))
    .sort_values("missing", ascending=False)
)
missing

,missing,pct
sustainability_badges,39267,92.01
buy_box_availability,14653,34.34
current/discounted_price,11749,27.53
delivery_details,11720,27.46
bought_in_last_month,3217,7.54
product_url,2069,4.85
rating,1024,2.40
number_of_reviews,1024,2.40
title,0,0.00
price_on_variant,0,0.00


## 4. Sample non-null values per column

In [5]:
for col in df.columns:
    sample = df[col].dropna().astype(str).head(2).tolist()
    print(f"{col:30s} → {sample}")

title                          → ['BOYA BOYALINK 2 Wireless Lavalier Microphone for iPhone Camera Android, Mini Lapel Micophone Wireless, 48 KHz 24 Bit, 6mm Mic, 1000ft, 30h Use, Noise Cancelling, Clip on Mic USB-C/Lightning/3.5mm TRS', 'LISEN USB C to Lightning Cable, 240W 4 in 1 Charging Cable 6.6FT, Chubby USB A/C to C/Lightning with Light for iPhone 16e 15 14 Pro/MacBook Air 17/iPad/Samsung/Switch 2, Multi Chargers for All Devices']
rating                         → ['4.6 out of 5 stars', '4.3 out of 5 stars']
number_of_reviews              → ['375', '2,457']
bought_in_last_month           → ['300+ bought in past month', '6K+ bought in past month']
current/discounted_price       → ['89.68', '9.99']
price_on_variant               → ['basic variant price: 2.4GHz', 'basic variant price: nan']
listed_price                   → ['$159.00', '$15.99']
is_best_seller                 → ['No Badge', 'No Badge']
is_sponsored                   → ['Sponsored', 'Sponsored']
is_couponed            

## 5. Format check — string columns that should be numeric

In [6]:
patterns = {
    "rating":              r"out of",
    "number_of_reviews":   r",|K|M",
    "bought_in_last_month":r"K\+|M\+|bought",
    "listed_price":        r"\$",
    "is_couponed":         r"%",
}
for col, pat in patterns.items():
    if col in df.columns:
        match_rate = df[col].dropna().astype(str).str.contains(pat).mean()
        print(f"{col:25s} matches '{pat:10s}'  in {match_rate:.0%} of non-null rows")

rating                    matches 'out of    '  in 100% of non-null rows
number_of_reviews         matches ',|K|M     '  in 37% of non-null rows
bought_in_last_month      matches 'K\+|M\+|bought'  in 82% of non-null rows
listed_price              matches '\$        '  in 29% of non-null rows
is_couponed               matches '%         '  in 2% of non-null rows


## 6. Duplicate analysis (Amazon search results often repeat titles across pages)

In [7]:
dup_total = df.duplicated().sum()
dup_titles = df["title"].duplicated().sum()
print(f"Exact duplicate rows : {dup_total:,}")
print(f"Duplicate titles     : {dup_titles:,}  ({dup_titles/len(df):.1%})")

Exact duplicate rows : 0
Duplicate titles     : 33,867  (79.4%)


## 7. Snapshot date sanity

In [8]:
df["collected_at"].value_counts().head()

collected_at
2025-08-21 11:14:29    33
2025-08-21 11:15:00    33
2025-08-21 11:15:31    33
2025-08-21 11:15:58    33
2025-08-21 11:16:09    33
Name: count, dtype: int64

## 8. Findings → cleaning plan
- Parse `rating` (`"4.6 out of 5 stars"` → 4.6).
- Parse `number_of_reviews` and `bought_in_last_month` shorthand (`"6K+"`, `"1,234"`).
- Parse `listed_price` (strip `$`, comma).
- `current/discounted_price` is already numeric-ish, has 27.5% missing → keep as `selling_price` fallback only when present.
- Convert badge fields (`is_best_seller`, `is_sponsored`, `buy_box_availability`) → booleans.
- Extract coupon % from `is_couponed`.
- Drop noise columns: `image_url`, `product_url`, `price_on_variant`, `delivery_details`, `sustainability_badges`.
- Dedupe by `title` (Amazon scrape ~85% duplicates across page pagination).
- Engineer: `selling_price`, `discount_pct`, `brand` (first token of title), `price_tier`, `rating_bucket`, `log_reviews`.